# 05 - Machine Learning

**Objectif**: Prédire quels clients vont churner

**Etapes**:  <br>
    1. Préparer les features (train/test split + normalisation)  <br>
    2. Entraîner deux modèles (Logistic Regression + Random Forest)  <br>
    3. Evaluer avec F1-score et AUC-ROC  <br>
    4. Interpréter avec feature importance  <br>
    5. Prédire sur un nouveau client  <br>
    6. Sauvegarder le modèle  <br>


**Métriques**: AUC-ROC + F1-score (pas l'accuracy - classes déséquilibrées)  <br>
**Fonctions utilisées**: `src/features.py`,`src/viz.py`

In [2]:
# Import
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import pickle, os
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

from src.features import (
    prepare_ml_data,
    predict_new_cient
)


## 1. Chargement et préparation des données

In [3]:
df_ml = pd.read_csv('../data/processed/telco_clean.csv')
print(f"Données chargées: {df_ml.shape}")
print(f"Features disponibles ({df_ml.shape[1]-1}) : {df_ml.drop('Churn', axis=1).columns.tolist()}")

Données chargées: (7043, 27)
Features disponibles (26) : ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [5]:
# prepare_ml_data (src/features.py)
#   1. Sépare X et y
#   2. Split stratifié train/test (80/20)
#   3. StandardScaler sur les colonnes numériques (fit sur train uniquement)

data = prepare_ml_data(df_ml, target='Churn', test_size=0.2, random_state=42)

X_train = data["X_train"]
X_test  = data['X_test']
y_train = data['y_train']
y_test  = data['y_test']
scaler  = data['scaler']
feature_names = data['feature_names']

print(f"Train: {X_train.shape} - Taux churn : {y_train.mean()*100:.1f}%")
print(f"Test: {X_test.shape} - Taux churn : {y_test.mean()*100:1f}")

Train: (5634, 26) - Taux churn : 26.5%
Test: (1409, 26) - Taux churn : 26.543648


## 2. Modèle 1 - Régression Logistique (baseline)

La régression logistique sert à prédire une probabilité pour un événement binaire: churn / pas churn, fraude / pas fraude, spam / pas spam, etc.  
Elle prédit la probabilité d'appartenir à la classe 1 grâce à une fonction sigmoïde.  
Ce n'est pas une régression, c'est un classifieur binaire.

**Le but**:  
On lui donne des features (tenure, MonthlyCharges, etc.), il renvoie:
* une probabilité entre 0 et 1  
* puis une classe (0 ou 1) selon un seuil (souvent 0.5)  

**AUC-ROC**:  
L'AUC-ROC mesure la capacité du modèle à séparer les classes (0 = No churn, 1 = Yes churn).  
Plus l'AUC est proche de 1, meilleur est le modèle

1) Le ROC : courbe des vrais positifs vs faux positifs:  
La courbe ROC (Receiver Operating Characteristic) trace:  
* axe Y: TPR = True Positif Rate
* axe X: FPR = False Positif Rate  
Et cela pour tous les seuils possibles (0 -> 1).  
Elle montre comment le modèle se comporte selon le seuil.

2) L'AUC: l'aire sous la courbe  
L'AUC = Area Under the Curve. C'est un nombre entre 0 et 1.
*Interprétation*:  
0.5         => nul (devine au hasard)
0.6-0.7     => faible
0.7-0.8     => correct
0.8-0.9     => bon
0.9-1.0     => excelllent

L'AUC-ROC est mieux que l'accuracy, car:
* ne dépend pas du seuil
* gère très bien les datasets déséquilibré (comme le churn)
* mesure la séparabilité du modèle.  
Exemple: Si le modèle donne des probabilité plus hautes aux churners qu'aux non-churners, l'AUC est élevé.  
Si AUC = 0.87, dans 87% des cas, il donne une probabilité plus élevée à un churner qu'à un non-churner.

**precision**   : prédit le % de ceux qui appartiennent réellement au groupe.  
**recall**      : parmi les vrais groupe de cette population, n% sont correctement détectées.  
**f1-score**    : C'est l'équilibre entre precision et recall. Un score de 0.60 est correct pour un dataset déséquilibré.  
**macro avg**   : moyenne simple entre les deux classes, ne tient pas compte du déséquilibre, utile pour voir si la classe est sacrifiée.  
**weighted avg**: moyenne pondérée par le nombre d'échantillon, reflète la performance gloable.

In [6]:
# Modèle de référence: simple, rapide, très interprétable
# Bon pour établir un AUC-ROC minimal à battre

lr = LogisticRegression(random_state=42, max_iter=1000, C=1.0)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test) # donne la prédiction finale du modèle en renvoyant la classe (0 ou 1) 
y_prob_lr = lr.predict_proba(X_test)[:, 1]  # récupère la probabilité de churn pour chaque client en renvoyant la probabilité d'être dans la classe 1

print("MODELE 1: Régression Logistique")
print(classification_report(y_test, y_pred_lr, target_names=['Non-Churn', 'Churn']))
print(f"AUC-ROC : {roc_auc_score(y_test, y_prob_lr):.4f}")

MODELE 1: Régression Logistique
              precision    recall  f1-score   support

   Non-Churn       0.85      0.89      0.87      1035
       Churn       0.65      0.56      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.74      1409
weighted avg       0.80      0.80      0.80      1409

AUC-ROC : 0.8423


Avec un AUC-ROC de 0.84, dans 84% des paires (un churner, un non-churner), le modèle attribue une probabilité plus élevé au churner qu'au non churner.  

## 3. Modèle 2 - Random Forest

Un **Random Forest** est un ensemble de beaucoup d'arbres de décision, chacun entraîné sur un échantillon différent.  
Le modèle final vote, ce qui donne un classifieur robuste, stable et performant.

#### Principe : une forêt d'arbres  
Un Random Forest = N arbres de décision (souvent 100 à 500).  
Chaque  arbre:  
* voit un sous-ensemble aléatoire des données
* voit un sous-ensemble aléatoire des features
* apprend une règle de décision différente

Ensuite:, on fait un vote majoritaire:
* pour la classification -> la classe la plus votée
* pour la probabilité -> moyenne des probabilité des arbres

#### Pourquoi c'est puissant ?
* il capture les intercations complexes entre variables
* il gère très bien les non-linéarités
* il est robuste au bruit
* il ne nécessite aucun scaling (contrairement à la régression logistique)
* il gère les outliers sans problème
* il donne des feature importances très utiles.

In [7]:
## clas_weight='balanced' : compense le déséquilibre Yes/No
# sans cela, le modèle favorise la classe majoritaire (Non-Churn)

rf = RandomForestClassifier(
    n_estimators=100, # nbr d'arbre dans la forêt
    random_state=42, # graine aléaoire
    n_jobs=-1, # indique combien de coeurs CPU utiliser
    class_weight='balanced' # améliore le recall et le F1-score de la classe churn
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("MODELE 2: Random Forest")
print(classification_report(y_test, y_pred_rf, target_names=['Non-Churn', 'Churn']))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob_rf):.4f}")

MODELE 2: Random Forest
              precision    recall  f1-score   support

   Non-Churn       0.83      0.89      0.86      1035
       Churn       0.62      0.49      0.55       374

    accuracy                           0.78      1409
   macro avg       0.72      0.69      0.70      1409
weighted avg       0.77      0.78      0.78      1409

AUC-ROC: 0.8242


* Accuracy = 0.78
* AUC-ROC = 0.8242  
L'AUC-ROC montre que le modèle distingue correctement les churners des non-churners: dans 82% des paires, il attribue une probabilité plus élevé au chruner.

Le Random Forest obtient un AUC-ROC de 0.8242, ce qui signifie qu'il attribue une probabilité plus élevée au churner dans 82% des paires churn/non-churn.  
Il performe très bien sur les non-churners (recall = 0.89), mais détecte moins bien les churners (recall = 0.49).  
Gloablement, le modèle est robuste, mais moins efficace que la régression logistique pourn identifier les clients à risque.